# 01 - Setup Check

Run this first. It confirms your catalog is set and the shared AkzoNobel coatings data is loaded. If anything is missing, it tells you exactly what to run.

The shared data setup lives in `data/` (see `data/README.md`). On a Vocareum lab it is usually pre-created by the lab setup script; this notebook just verifies it.

In [ ]:
# Catalog defaults to your assigned lab catalog. Nothing is hardcoded.
default_catalog = spark.sql("SELECT current_catalog()").first()[0]
dbutils.widgets.text("catalog", default_catalog, "Unity Catalog")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "LLM endpoint")
catalog = dbutils.widgets.get("catalog")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
print("Catalog:", catalog)
print("LLM endpoint:", llm_endpoint)

In [ ]:
# Check the expected schemas exist.
expected_schemas = ["akzo_finance", "akzo_scm", "akzo_commercial", "akzo_docs"]
present = {r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {catalog}").collect()}
missing = [s for s in expected_schemas if s not in present]
if missing:
    print("MISSING schemas:", missing)
    print("Run the shared data setup first — see data/README.md (generators + load_to_uc.py).")
else:
    print("OK — all expected schemas present:", expected_schemas)

In [ ]:
# Spot-check a core table has rows.
if not missing:
    n = spark.sql(f"SELECT count(*) AS n FROM {catalog}.akzo_finance.margin_actuals").first()["n"]
    print(f"akzo_finance.margin_actuals rows: {n}")
    print("Setup looks good — open the first L100 notebook." if n > 0 else "Table is empty — re-run load_to_uc.py.")

In [ ]:
# Optional: confirm the LLM endpoint answers (used by SQL AI Functions and agents).
try:
    r = spark.sql(f"SELECT ai_query('{llm_endpoint}', 'Reply with the single word: ready') AS r").first()["r"]
    print("LLM endpoint responded:", r)
except Exception as e:
    print("LLM endpoint check failed — confirm the endpoint name in the widget. Detail:", str(e)[:300])